In [14]:
import numpy as np
metodos=("jacobi","gauss_seidel")

In [15]:
# Criterio suficiente pero no necesario de convergencia: diagonal dominante por filas
def convergencia_suficiente(A:np.array):
    n=A.shape[0]
    for i in range(n):
        suma=np.sum(np.abs(A[i,:]))-np.abs(A[i,i])
        if np.abs(A[i,i])<=suma:
            return False
    return True

# Criterio necesario de convergencia: diagonal mayor por filas
def convergencia_necesaria(A:np.array):
    n=A.shape[0]
    for i in range(n):
        for j in range(n):
            if i!=j and np.abs(A[i,j])>np.abs(A[i,i]):
                return False
    return True

# Arreglar matriz teniendo en cuenta que el criterio de convergencia necesario pero no suficiente
def arreglar(A:np.array,b:np.array):
    n=A.shape[0]
    for i in range(n):
        for j in range(n):
            if i!=j and np.abs(A[i,j])>np.abs(A[i,i]):
                A[[i,j]]=A[[j,i]]
                b[[i,j]]=b[[j,i]]
    return A,b

# Método de Jacobi (itera sobre x_old)
def jacobi(A:np.array,b:np.array,tol:float,max_iter:int):
    n=A.shape[0]
    x0=np.zeros(n)
    x=x0.copy()
    for k in range(max_iter):
        x_old=x.copy()
        for i in range(n):
            suma=np.dot(A[i,:],x_old)-A[i,i]*x_old[i]
            x[i]=(b[i]-suma)/A[i,i]
        if np.linalg.norm(x-x_old,ord=np.inf)<tol:
            return x
    return x

# Método de Gauss-Seidel (itera sobre x)
def gauss_seidel(A:np.array,b:np.array,tol:float,max_iter:int):
    n=A.shape[0]
    x0=np.zeros(n)
    x=x0.copy()
    for k in range(max_iter):
        x_old=x.copy()
        for i in range(n):
            suma=np.dot(A[i,:],x)-A[i,i]*x[i]
            x[i]=(b[i]-suma)/A[i,i]
        if np.linalg.norm(x-x_old,ord=np.inf)<tol:
            return x
    return x

# Criterio de convergencia radio espectral del método iterativo
def convergencia_rho(A:np.array,metodo:str):
    def radio_espectral(A:np.array,metodo:str):
        n=A.shape[0]
        if metodo=="jacobi":
            D=np.diag(np.diag(A))
            R=A-D
            T=-np.linalg.inv(D)@R
        elif metodo=="gauss_seidel":
            D=np.diag(np.diag(A))
            L=np.tril(A)-D
            U=np.triu(A)-D
            T=np.linalg.inv(D+L)@U
        else:
            raise ValueError("Use jacobi o gauss_seidel")
        eigenv,_=np.linalg.eig(T)
        return max(np.abs(eigenv))
    return radio_espectral(A,metodo)<1

In [16]:
# Ejemplo 1
A1=np.array([
    [4,-1,0,0,0,0,0,0],
    [-1,4,-1,0,0,0,0,0],
    [0,-1,4,-1,0,0,0,0],
    [0,0,-1,4,-1,0,0,0],
    [0,0,0,-1,4,-1,0,0],
    [0,0,0,0,-1,4,-1,0],
    [0,0,0,0,0,-1,4,-1],
    [0,0,0,0,0,0,-1,4],])
b1=np.array([1,2,3,4,5,6,7,8])

for metodo in metodos:
    print(f"Método: {metodo}")
    if convergencia_suficiente(A1): print("Criterio suficiente de convergencia: Sí")
    else: print("Criterio suficiente de convergencia: No")
    if convergencia_necesaria(A1): print("Criterio necesario de convergencia: Sí")
    else: print("Criterio necesario de convergencia: No")
    if convergencia_rho(A1,metodo): 
        print("Criterio de convergencia por radio espectral: Sí")
        if metodo=="jacobi": x=jacobi(A1,b1,tol=1e-6,max_iter=100)
        else: x=gauss_seidel(A1,b1,tol=1e-6,max_iter=100)
        print(f"Solución aproximada: {x}\n",
              f"Confirmación: {A1@x}\n")
    else: print("Criterio de convergencia por radio espectral: No\n")

Método: jacobi
Criterio suficiente de convergencia: Sí
Criterio necesario de convergencia: Sí
Criterio de convergencia por radio espectral: Sí
Solución aproximada: [0.49988877 0.99955558 1.49833456 1.99378395 2.47680283 2.91342887
 3.17691407 2.79422838]
 Confirmación: [0.99999949 1.99999898 2.9999987  3.99999842 4.99999851 5.9999986
 6.99999902 7.99999944]

Método: gauss_seidel
Criterio suficiente de convergencia: Sí
Criterio necesario de convergencia: Sí
Criterio de convergencia por radio espectral: Sí
Solución aproximada: [0.49988883 0.99955588 1.49833508 1.99378463 2.47680352 2.91342951
 3.17691454 2.79422863]
 Confirmación: [0.99999943 1.99999963 2.9999998  3.9999999  4.99999996 5.99999999
 7.         8.        ]



In [17]:
# Ejemplo 2
A2=np.array([
    [1,-4,0,0,0,0,0,0],
    [-4,1,-4,0,0,0,0,0],
    [0,-4,1,-4,0,0,0,0],
    [0,0,-4,1,-4,0,0,0],
    [0,0,0,-4,1,-4,0,0],
    [0,0,0,0,-4,1,-4,0],
    [0,0,0,0,0,-4,1,-4],
    [0,0,0,0,0,0,-4,1],])
b2=np.array([8,7,6,5,4,3,2,1])

for metodo in metodos:
    print(f"Método: {metodo}")
    if convergencia_suficiente(A2): print("Criterio suficiente de convergencia: Sí")
    else: print("Criterio suficiente de convergencia: No")
    if convergencia_necesaria(A2): print("Criterio necesario de convergencia: Sí")
    else: print("Criterio necesario de convergencia: No")
    if convergencia_rho(A2,metodo): 
        print("Criterio de convergencia por radio espectral: Sí")
        if metodo=="jacobi": x=jacobi(A2,b2,tol=1e-6,max_iter=100)
        else: x=gauss_seidel(A2,b2,tol=1e-6,max_iter=100)
        print(f"Solución aproximada: {x}\n",
              f"Confirmación: {A2@x}\n")
    else: print("Criterio de convergencia por radio espectral: No\n")

Método: jacobi
Criterio suficiente de convergencia: No
Criterio necesario de convergencia: No
Criterio de convergencia por radio espectral: No

Método: gauss_seidel
Criterio suficiente de convergencia: No
Criterio necesario de convergencia: No
Criterio de convergencia por radio espectral: No



In [18]:
A2=arreglar(A2,b2)[0]
b2=arreglar(A2,b2)[1]
for metodo in metodos:
    print(f"Método: {metodo}")
    if convergencia_suficiente(A2): print("Criterio suficiente de convergencia: Sí")
    else: print("Criterio suficiente de convergencia: No")
    if convergencia_necesaria(A2): print("Criterio necesario de convergencia: Sí")
    else: print("Criterio necesario de convergencia: No")
    if convergencia_rho(A2,metodo): 
        print("Criterio de convergencia por radio espectral: Sí")
        if metodo=="jacobi": x=jacobi(A2,b2,tol=1e-6,max_iter=100)
        else: x=gauss_seidel(A2,b2,tol=1e-6,max_iter=100)
        print(f"Solución aproximada: {x}\n",
              f"Confirmación: {A2@x}\n")
    else: print("Criterio de convergencia por radio espectral: No\n")

Método: jacobi
Criterio suficiente de convergencia: No
Criterio necesario de convergencia: No
Criterio de convergencia por radio espectral: No

Método: gauss_seidel
Criterio suficiente de convergencia: No
Criterio necesario de convergencia: No
Criterio de convergencia por radio espectral: No



In [19]:
# Ejemplo 3
A3=np.array([
    [5,-4,0,0,0,0,0,0],
    [-4,5,-4,0,0,0,0,0],
    [0,-4,5,-4,0,0,0,0],
    [0,0,-4,5,-4,0,0,0],
    [0,0,0,-4,5,-4,0,0],
    [0,0,0,0,-4,5,-4,0],
    [0,0,0,0,0,-4,5,-4],
    [0,0,0,0,0,0,-4,5],])
b3=np.array([8,7,6,5,4,3,2,1])

for metodo in metodos:
    print(f"Método: {metodo}")
    if convergencia_suficiente(A3): print("Criterio suficiente de convergencia: Sí")
    else: print("Criterio suficiente de convergencia: No")
    if convergencia_necesaria(A3): print("Criterio necesario de convergencia: Sí")
    else: print("Criterio necesario de convergencia: No")
    if convergencia_rho(A3,metodo): 
        print("Criterio de convergencia por radio espectral: Sí")
        if metodo=="jacobi": x=jacobi(A3,b3,tol=1e-6,max_iter=100)
        else: x=gauss_seidel(A3,b3,tol=1e-6,max_iter=100)
        print(f"Solución aproximada: {x}\n",
              f"Confirmación: {A3@x}\n")
    else: print("Criterio de convergencia por radio espectral: No\n")

Método: jacobi
Criterio suficiente de convergencia: No
Criterio necesario de convergencia: Sí
Criterio de convergencia por radio espectral: No

Método: gauss_seidel
Criterio suficiente de convergencia: No
Criterio necesario de convergencia: Sí
Criterio de convergencia por radio espectral: No

